<p style="color:white; font-weight:bold; font-size:17px;">8-Puzzle Solver</p>

<p style="color:white; font-weight:bold; font-size:17px;">Required Initializations</p>

In [16]:
from __future__ import annotations

from dataclasses import dataclass
from collections import deque
from heapq import heappop, heappush
from math import sqrt
from time import perf_counter
from typing import Callable, Dict, List, Optional, Sequence, Set, Tuple

State = Tuple[int, ...]
GOAL_STATE: State = (0, 1, 2, 3, 4, 5, 6, 7, 8)
MOVE_ORDER: Tuple[str, ...] = ("Up", "Down", "Left", "Right")


@dataclass
class SearchResult:
    algorithm: str
    initial_state: State
    goal_state: State
    found: bool
    path_to_goal: List[str]
    states_path: List[State]
    cost_of_path: int
    nodes_expanded: int
    search_depth: int
    running_time: float
    max_frontier_size: int
    message: str = ""


class PuzzleError(ValueError):
    pass

<div style="color:white;">
  <div style="font-size:32px; font-weight:400; color:white; margin-bottom:24px;">
    2) Common helper functions
  </div>

  <div style="font-size:16px; color:white; margin-bottom:12px;">
    This section contains the shared tools used by all algorithms:
  </div>

  <ul style="font-size:16px; color:white; margin-top:0;">
    <li>parsing the input state</li>
    <li>printing the board</li>
    <li>checking solvability</li>
    <li>generating neighbors</li>
    <li>rebuilding the solution path</li>
    <li>computing heuristics for A*</li>
  </ul>
</div>

In [17]:
def parse_state(values: Sequence[int] | str) -> State:
    if isinstance(values, str):
        cleaned = values.replace(" ", "")
        parts = cleaned.split(",")
        nums = tuple(int(x) for x in parts)
    else:
        nums = tuple(int(x) for x in values)

    if len(nums) != 9:
        raise PuzzleError("State must contain exactly 9 values.")
    if set(nums) != set(range(9)):
        raise PuzzleError("State must contain each number from 0 to 8 exactly once.")
    return nums


def board_to_string(state: State) -> str:
    rows = []
    for i in range(0, 9, 3):
        row = []
        for value in state[i:i + 3]:
            row.append("_" if value == 0 else str(value))
        rows.append(" ".join(f"{cell:>2}" for cell in row))
    return "\n".join(rows)


def goal_position(tile: int) -> Tuple[int, int]:
    index = GOAL_STATE.index(tile)
    return divmod(index, 3)


def manhattan_distance(state: State) -> float:
    distance = 0
    for index, tile in enumerate(state):
        if tile == 0:
            continue
        row, col = divmod(index, 3)
        goal_row, goal_col = goal_position(tile)
        distance += abs(row - goal_row) + abs(col - goal_col)
    return float(distance)


def euclidean_distance(state: State) -> float:
    distance = 0.0
    for index, tile in enumerate(state):
        if tile == 0:
            continue
        row, col = divmod(index, 3)
        goal_row, goal_col = goal_position(tile)
        distance += sqrt((row - goal_row) ** 2 + (col - goal_col) ** 2)
    return distance


def inversion_count(state: State) -> int:
    values = [x for x in state if x != 0]
    inv = 0
    for i in range(len(values)):
        for j in range(i + 1, len(values)):
            if values[i] > values[j]:
                inv += 1
    return inv


def is_solvable(state: State) -> bool:
    # For a 3x3 puzzle (odd width), the puzzle is solvable if the number of inversions is even.
    return inversion_count(state) % 2 == 0


def neighbors(state: State) -> List[Tuple[str, State]]:
    zero_index = state.index(0)
    row, col = divmod(zero_index, 3)
    result: List[Tuple[str, State]] = []

    def swap_and_store(move: str, target_index: int) -> None:
        board = list(state)
        board[zero_index], board[target_index] = board[target_index], board[zero_index]
        result.append((move, tuple(board)))

    for move in MOVE_ORDER:
        if move == "Up" and row > 0:
            swap_and_store(move, zero_index - 3)
        elif move == "Down" and row < 2:
            swap_and_store(move, zero_index + 3)
        elif move == "Left" and col > 0:
            swap_and_store(move, zero_index - 1)
        elif move == "Right" and col < 2:
            swap_and_store(move, zero_index + 1)

    return result


def reconstruct_path(
    came_from: Dict[State, Optional[State]],
    action_from: Dict[State, Optional[str]],
    goal: State,
) -> Tuple[List[str], List[State]]:
    actions: List[str] = []
    states: List[State] = []
    current: Optional[State] = goal

    while current is not None:
        states.append(current)
        action = action_from.get(current)
        if action is not None:
            actions.append(action)
        current = came_from.get(current)

    actions.reverse()
    states.reverse()
    return actions, states


def failure_result(algorithm: str, initial_state: State, start_time: float, message: str) -> SearchResult:
    return SearchResult(
        algorithm=algorithm,
        initial_state=initial_state,
        goal_state=GOAL_STATE,
        found=False,
        path_to_goal=[],
        states_path=[initial_state],
        cost_of_path=0,
        nodes_expanded=0,
        search_depth=0,
        running_time=perf_counter() - start_time,
        max_frontier_size=0,
        message=message,
    )

<div style="color:#ffffff !important;">
  <div style="font-size:26px; font-weight:400; color:#ffffff !important; margin-bottom:16px;">
    3) BFS — Breadth-First Search
  </div>

  <div style="font-size:20px; font-weight:400; color:#ffffff !important; margin-top:12px; margin-bottom:8px;">
    What does BFS do?
  </div>
  <div style="font-size:16px; color:#ffffff !important; margin-bottom:14px;">
    BFS explores the search tree <b>level by level</b>.
  </div>

  <div style="font-size:20px; font-weight:400; color:#ffffff !important; margin-top:12px; margin-bottom:8px;">
    Data structure used
  </div>
  <ul style="font-size:16px; color:#ffffff !important; margin-top:0; margin-bottom:14px;">
    <li><b>Queue</b></li>
  </ul>

  <div style="font-size:20px; font-weight:400; color:#ffffff !important; margin-top:12px; margin-bottom:8px;">
    Main idea
  </div>
  <ul style="font-size:16px; color:#ffffff !important; margin-top:0; margin-bottom:14px;">
    <li>Put the initial state in the queue.</li>
    <li>Remove the front state.</li>
    <li>Expand its neighbors.</li>
    <li>Add new neighbors to the back of the queue.</li>
    <li>Repeat until the goal is found.</li>
  </ul>

  <div style="font-size:20px; font-weight:400; color:#ffffff !important; margin-top:12px; margin-bottom:8px;">
    Important property
  </div>
  <div style="font-size:16px; color:#ffffff !important;">
    Because every move has the same cost , <b>BFS gives the shortest path in number of moves</b>.
  </div>
</div>

In [18]:
def bfs(initial_state: State) -> SearchResult:
    algorithm = "BFS"
    start_time = perf_counter()

    if not is_solvable(initial_state):
        return failure_result(algorithm, initial_state, start_time, "This puzzle is not solvable.")

    frontier = deque([initial_state])       # Queue
    frontier_set = {initial_state}          # Fast membership check
    explored: Set[State] = set()

    came_from: Dict[State, Optional[State]] = {initial_state: None}
    action_from: Dict[State, Optional[str]] = {initial_state: None}
    depth: Dict[State, int] = {initial_state: 0}

    nodes_expanded = 0
    max_frontier_size = 1

    while frontier:
        state = frontier.popleft()
        frontier_set.remove(state)

        if state == GOAL_STATE:
            path, states_path = reconstruct_path(came_from, action_from, state)
            return SearchResult(
                algorithm=algorithm,
                initial_state=initial_state,
                goal_state=GOAL_STATE,
                found=True,
                path_to_goal=path,
                states_path=states_path,
                cost_of_path=len(path),
                nodes_expanded=nodes_expanded,
                search_depth=depth[state],
                running_time=perf_counter() - start_time,
                max_frontier_size=max_frontier_size,
            )

        explored.add(state)
        nodes_expanded += 1

        for move, neighbor in neighbors(state):
            if neighbor not in frontier_set and neighbor not in explored:
                came_from[neighbor] = state
                action_from[neighbor] = move
                depth[neighbor] = depth[state] + 1
                frontier.append(neighbor)
                frontier_set.add(neighbor)

        max_frontier_size = max(max_frontier_size, len(frontier))

    return failure_result(algorithm, initial_state, start_time, "No solution found.")

<div style="color:#ffffff !important;">
  <div style="font-size:26px; font-weight:400; color:#ffffff !important; margin-bottom:16px;">
    4) DFS — Depth-First Search
  </div>

  <div style="font-size:20px; font-weight:400; color:#ffffff !important; margin-top:12px; margin-bottom:8px;">
    What does DFS do?
  </div>
  <div style="font-size:16px; color:#ffffff !important; margin-bottom:14px;">
    DFS goes <b>deep into one branch first</b>, then backtracks when needed.
  </div>

  <div style="font-size:20px; font-weight:400; color:#ffffff !important; margin-top:12px; margin-bottom:8px;">
    Data structure used
  </div>
  <ul style="font-size:16px; color:#ffffff !important; margin-top:0; margin-bottom:14px;">
    <li><b>Stack</b></li>
  </ul>

  <div style="font-size:20px; font-weight:400; color:#ffffff !important; margin-top:12px; margin-bottom:8px;">
    Main idea
  </div>
  <ul style="font-size:16px; color:#ffffff !important; margin-top:0; margin-bottom:14px;">
    <li>Put the initial state in the stack.</li>
    <li>Pop the top state.</li>
    <li>Expand its neighbors.</li>
    <li>Push new neighbors onto the stack.</li>
    <li>Continue until the goal is found.</li>
  </ul>

  <div style="font-size:20px; font-weight:400; color:#ffffff !important; margin-top:12px; margin-bottom:8px;">
    Important note
  </div>
  <div style="font-size:16px; color:#ffffff !important; margin-bottom:10px;">
    DFS is usually memory-light, but it <b>does not guarantee the shortest path</b>.
  </div>
  <div style="font-size:16px; color:#ffffff !important;">
    In 8-puzzle, DFS can sometimes expand <b>many more nodes</b> than BFS or A* depending on the move order.
  </div>
</div>

In [19]:
def dfs(initial_state: State) -> SearchResult:
    algorithm = "DFS"
    start_time = perf_counter()

    if not is_solvable(initial_state):
        return failure_result(algorithm, initial_state, start_time, "This puzzle is not solvable.")

    frontier: List[State] = [initial_state]  # Stack
    frontier_set = {initial_state}
    explored: Set[State] = set()

    came_from: Dict[State, Optional[State]] = {initial_state: None}
    action_from: Dict[State, Optional[str]] = {initial_state: None}
    depth: Dict[State, int] = {initial_state: 0}

    nodes_expanded = 0
    max_frontier_size = 1

    while frontier:
        state = frontier.pop()
        frontier_set.remove(state)

        if state == GOAL_STATE:
            path, states_path = reconstruct_path(came_from, action_from, state)
            return SearchResult(
                algorithm=algorithm,
                initial_state=initial_state,
                goal_state=GOAL_STATE,
                found=True,
                path_to_goal=path,
                states_path=states_path,
                cost_of_path=len(path),
                nodes_expanded=nodes_expanded,
                search_depth=depth[state],
                running_time=perf_counter() - start_time,
                max_frontier_size=max_frontier_size,
            )

        explored.add(state)
        nodes_expanded += 1

        # Reverse push order so the effective expansion order remains Up, Down, Left, Right.
        for move, neighbor in reversed(neighbors(state)):
            if neighbor not in frontier_set and neighbor not in explored:
                came_from[neighbor] = state
                action_from[neighbor] = move
                depth[neighbor] = depth[state] + 1
                frontier.append(neighbor)
                frontier_set.add(neighbor)

        max_frontier_size = max(max_frontier_size, len(frontier))

    return failure_result(algorithm, initial_state, start_time, "No solution found.")

<div style="color:#ffffff !important;">
  <div style="font-size:26px; font-weight:400; color:#ffffff !important; margin-bottom:16px;">
    5) A* Search
  </div>

  <div style="font-size:20px; font-weight:400; color:#ffffff !important; margin-top:12px; margin-bottom:8px;">
    What does A* do?
  </div>
  <div style="font-size:16px; color:#ffffff !important; margin-bottom:10px;">
    A* is an <b>informed search algorithm</b>.
  </div>
  <div style="font-size:16px; color:#ffffff !important; margin-bottom:10px;">
    It chooses the node with the smallest:
  </div>

  <div style="font-size:18px; color:#ffffff !important; margin:12px 0 14px 0;">
    <b>f(n) = g(n) + h(n)</b>
  </div>

  <div style="font-size:16px; color:#ffffff !important; margin-bottom:8px;">
    Where:
  </div>
  <ul style="font-size:16px; color:#ffffff !important; margin-top:0; margin-bottom:14px;">
    <li><b>g(n)</b> = cost from the start node to the current node</li>
    <li><b>h(n)</b> = estimated remaining cost to the goal</li>
    <li><b>f(n)</b> = total estimated cost</li>
  </ul>

  <div style="font-size:20px; font-weight:400; color:#ffffff !important; margin-top:12px; margin-bottom:8px;">
    Data structure used
  </div>
  <ul style="font-size:16px; color:#ffffff !important; margin-top:0; margin-bottom:14px;">
    <li><b>Priority Queue / Min-Heap</b></li>
  </ul>

  <div style="font-size:20px; font-weight:400; color:#ffffff !important; margin-top:12px; margin-bottom:8px;">
    Heuristics used in this assignment
  </div>
  <ol style="font-size:16px; color:#ffffff !important; margin-top:0; margin-bottom:14px;">
    <li><b>Manhattan distance</b></li>
    <li><b>Euclidean distance</b></li>
  </ol>

  <div style="font-size:20px; font-weight:400; color:#ffffff !important; margin-top:12px; margin-bottom:8px;">
    Important note
  </div>
  <div style="font-size:16px; color:#ffffff !important; margin-bottom:10px;">
    If A* later finds a <b>cheaper path</b> to a state, that state can be updated and used again.
  </div>
  <div style="font-size:16px; color:#ffffff !important;">
    That is why this implementation can <b>reopen</b> a closed node when a better path appears.
  </div>
</div>

In [20]:
def a_star(initial_state: State, heuristic: Callable[[State], float], heuristic_name: str) -> SearchResult:
    algorithm = f"A* ({heuristic_name})"
    start_time = perf_counter()

    if not is_solvable(initial_state):
        return failure_result(algorithm, initial_state, start_time, "This puzzle is not solvable.")

    heap: List[Tuple[float, int, State]] = []
    counter = 0

    best_g: Dict[State, float] = {initial_state: 0.0}
    depth: Dict[State, int] = {initial_state: 0}
    came_from: Dict[State, Optional[State]] = {initial_state: None}
    action_from: Dict[State, Optional[str]] = {initial_state: None}

    closed: Set[State] = set()
    open_best_f: Dict[State, float] = {}

    nodes_expanded = 0
    max_frontier_size = 1

    start_f = heuristic(initial_state)
    heappush(heap, (start_f, counter, initial_state))
    open_best_f[initial_state] = start_f

    while heap:
        f_score, _, state = heappop(heap)

        # Skip stale heap entries
        current_best_f = open_best_f.get(state)
        if current_best_f is None or f_score != current_best_f:
            continue

        del open_best_f[state]

        if state == GOAL_STATE:
            path, states_path = reconstruct_path(came_from, action_from, state)
            return SearchResult(
                algorithm=algorithm,
                initial_state=initial_state,
                goal_state=GOAL_STATE,
                found=True,
                path_to_goal=path,
                states_path=states_path,
                cost_of_path=int(best_g[state]),
                nodes_expanded=nodes_expanded,
                search_depth=depth[state],
                running_time=perf_counter() - start_time,
                max_frontier_size=max_frontier_size,
            )

        closed.add(state)
        nodes_expanded += 1

        for move, neighbor in neighbors(state):
            tentative_g = best_g[state] + 1.0

            if tentative_g < best_g.get(neighbor, float("inf")):
                came_from[neighbor] = state
                action_from[neighbor] = move
                best_g[neighbor] = tentative_g
                depth[neighbor] = depth[state] + 1

                # Reopen a closed state if we found a cheaper path.
                if neighbor in closed:
                    closed.remove(neighbor)

                counter += 1
                new_f = tentative_g + heuristic(neighbor)
                heappush(heap, (new_f, counter, neighbor))
                open_best_f[neighbor] = new_f

        max_frontier_size = max(max_frontier_size, len(open_best_f))

    return failure_result(algorithm, initial_state, start_time, "No solution found.")

<div style="color:#ffffff !important;">
  <div style="font-size:26px; font-weight:400; color:#ffffff !important; margin-bottom:16px;">
    6) Unified solver + clean output printer
  </div>

  <div style="font-size:16px; color:#ffffff !important;">
    This section gives one function to call any required algorithm, plus a neat display function for the report.
  </div>
</div>

In [21]:
def solve(initial_state: State | Sequence[int] | str, algorithm: str) -> SearchResult:
    state = parse_state(initial_state)
    key = algorithm.strip().lower()

    if key == "bfs":
        return bfs(state)
    if key == "dfs":
        return dfs(state)
    if key in {"astar_manhattan", "a*_manhattan", "a*manhattan", "manhattan"}:
        return a_star(state, manhattan_distance, "Manhattan")
    if key in {"astar_euclidean", "a*_euclidean", "a*euclidean", "euclidean"}:
        return a_star(state, euclidean_distance, "Euclidean")

    raise PuzzleError("Algorithm must be one of: bfs, dfs, astar_manhattan, astar_euclidean.")


def print_result(result: SearchResult, show_trace: bool = True) -> None:
    print("=" * 60)
    print(f"Algorithm       : {result.algorithm}")
    print(f"Initial State   : {','.join(map(str, result.initial_state))}")
    print(f"Goal State      : {','.join(map(str, result.goal_state))}")
    print(f"Found           : {result.found}")

    if not result.found:
        print(f"Message         : {result.message}")
        print("=" * 60)
        return

    print(f"Path to Goal    : {result.path_to_goal}")
    print(f"Cost of Path    : {result.cost_of_path}")
    print(f"Nodes Expanded  : {result.nodes_expanded}")
    print(f"Search Depth    : {result.search_depth}")
    print(f"Running Time    : {result.running_time:.6f} seconds")
    print(f"Max Frontier    : {result.max_frontier_size}")

    if show_trace:
        print("\nTrace:")
        for step, state in enumerate(result.states_path):
            print(f"\nStep {step}")
            if step > 0:
                print(f"Move used: {result.path_to_goal[step - 1]}")
            print(board_to_string(state))

    print("=" * 60)

<div style="color:#ffffff !important;">
  <div style="font-size:26px; font-weight:400; color:#ffffff !important; margin-bottom:16px;">
    7) Sample run — BFS, DFS, A*
  </div>

  <div style="font-size:16px; color:#ffffff !important; margin-bottom:10px;">
    Example used:
  </div>

  <div style="font-size:16px; color:#ffffff !important;">
    <code style="color:#ffffff !important; background:transparent;">1,2,5,3,4,0,6,7,8</code>
  </div>
</div>`

In [22]:
# =========================
# BFS Test
# =========================
sample_state = "1,2,5,3,4,0,6,7,8"
bfs_result = solve(sample_state, "bfs")
print_result(bfs_result, show_trace=True)

print("\n" + "=" * 60 + "\n")


# =========================
# DFS Test
# =========================
dfs_demo_state = "3,1,2,0,4,5,6,7,8"
dfs_result = solve(dfs_demo_state, "dfs")
print_result(dfs_result, show_trace=True)

print("\n" + "=" * 60 + "\n")


# =========================
# A* Manhattan Test
# =========================
astar_manhattan_result = solve(sample_state, "astar_manhattan")
print_result(astar_manhattan_result, show_trace=True)

print("\n" + "=" * 60 + "\n")


# =========================
# A* Euclidean Test
# =========================
astar_euclidean_result = solve(sample_state, "astar_euclidean")
print_result(astar_euclidean_result, show_trace=True)

Algorithm       : BFS
Initial State   : 1,2,5,3,4,0,6,7,8
Goal State      : 0,1,2,3,4,5,6,7,8
Found           : True
Path to Goal    : ['Up', 'Left', 'Left']
Cost of Path    : 3
Nodes Expanded  : 10
Search Depth    : 3
Running Time    : 0.000064 seconds
Max Frontier    : 12

Trace:

Step 0
 1  2  5
 3  4  _
 6  7  8

Step 1
Move used: Up
 1  2  _
 3  4  5
 6  7  8

Step 2
Move used: Left
 1  _  2
 3  4  5
 6  7  8

Step 3
Move used: Left
 _  1  2
 3  4  5
 6  7  8


Algorithm       : DFS
Initial State   : 3,1,2,0,4,5,6,7,8
Goal State      : 0,1,2,3,4,5,6,7,8
Found           : True
Path to Goal    : ['Up']
Cost of Path    : 1
Nodes Expanded  : 1
Search Depth    : 1
Running Time    : 0.000027 seconds
Max Frontier    : 3

Trace:

Step 0
 3  1  2
 _  4  5
 6  7  8

Step 1
Move used: Up
 _  1  2
 3  4  5
 6  7  8


Algorithm       : A* (Manhattan)
Initial State   : 1,2,5,3,4,0,6,7,8
Goal State      : 0,1,2,3,4,5,6,7,8
Found           : True
Path to Goal    : ['Up', 'Left', 'Left']
Cost of P

<div style="color:#ffffff !important;">
  <div style="font-size:26px; font-weight:400; color:#ffffff !important; margin-bottom:16px;">
     Quick comparison table
  </div>

  <div style="font-size:16px; color:#ffffff !important;">
    This gives a compact comparison between the algorithms.
  </div>
</div>

In [23]:
comparison_results = [
    solve(sample_state, "bfs"),
    solve(dfs_demo_state, "dfs"),
    solve(sample_state, "astar_manhattan"),
    solve(sample_state, "astar_euclidean"),
]

print(f"{'Algorithm':<22} {'Cost':<6} {'Expanded':<10} {'Depth':<8} {'Time (s)':<12}")
print("-" * 64)

for r in comparison_results:
    print(f"{r.algorithm:<22} {r.cost_of_path:<6} {r.nodes_expanded:<10} {r.search_depth:<8} {r.running_time:<12.6f}")

Algorithm              Cost   Expanded   Depth    Time (s)    
----------------------------------------------------------------
BFS                    3      10         3        0.000057    
DFS                    1      1          1        0.000013    
A* (Manhattan)         3      3          3        0.000049    
A* (Euclidean)         3      3          3        0.000043    
